# Regime-Aware GBM — WTI Weekly Return Forecasting
## 4-State Oil Market Regime Detection

Extends the walk-forward GBM by detecting the dominant market regime each week
and feeding regime probabilities as extra features.

**Regimes:**
| Regime | Driver signals |
|--------|---------------|
| `supply`   | OPEC event day, disruption intensity, OPEC sentiment |
| `demand`   | S&P 500 return, EIA inventory draw/build |
| `macro`    | USD momentum, 10Y yield change |
| `risk_off` | Realised vol, FinBERT negative prob, negative news ratio |

**Key insight:** The GBM can learn to up-weight OPEC features in `supply` regimes
and ignore CS_DI in `risk_off` regimes where stale sentiment hurts.


## 1. Imports & Config

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..')); os.chdir('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance

from src.features.oil_regime import OilRegimeDetector

np.random.seed(42)

# ── Walk-forward parameters ────────────────────────────────────────────────────
INITIAL_TRAIN_WEEKS  = 104
STEP_WEEKS           = 13
CONFIDENCE_THRESHOLD = 1.0

# ── Model parameters ──────────────────────────────────────────────────────────
DECAY_RATE           = 0.995
VOL_WINDOW           = 8
MIN_REGIME_SAMPLES   = 25    # weeks needed to train a specialist (else global fallback)
GLOBAL_WEIGHT        = 0.15  # weight given to global model in blend (rest = specialists)

# ── Feature columns (39) ──────────────────────────────────────────────────────
FEATURE_COLS = [
    'headline_count_lag1', 'sentiment_mean_lag1', 'sentiment_std_lag1',
    'sentiment_min_lag1', 'sentiment_max_lag1', 'negative_ratio_lag1',
    'positive_ratio_lag1', 'log_headline_count_lag1',
    'opec_event_day_lag1', 'opec_decision_day_lag1', 'opec_sentiment_lag1',
    'disruption_event_day_lag1', 'disruption_intensity_lag1',
    'finbert_positive_mean_lag1', 'finbert_negative_mean_lag1', 'finbert_neutral_mean_lag1',
    'realized_vol_lag1',
    'regime_0_lag1', 'regime_1_lag1', 'regime_2_lag1',
    'hmm_regime_0_lag1', 'hmm_regime_1_lag1', 'hmm_regime_2_lag1',
    'inventory_chg_kb_lag1', 'inventory_surprise_kb_lag1',
    'production_chg_kbd_lag1', 'imports_chg_kbd_lag1',
    'refinery_util_pct_lag1', 'refinery_util_chg_pct_lag1',
    'usd_ret_pct_lag1', 'usd_mom_4w_lag1', 'spx_ret_pct_lag1', 'yield_10y_chg_lag1',
    'broad_cs_di_lag1', 'opec_cs_di_lag1', 'disruption_cs_di_lag1',
    'broad_csi_v_lag1', 'opec_csi_v_lag1', 'disruption_csi_v_lag1',
]

REGIME_NAMES = ['supply', 'demand', 'macro', 'risk_off']
TARGET_COL   = 'close_pct_change'

def _make_model(**kw):
    defaults = dict(loss='absolute_error', max_depth=4, learning_rate=0.05,
                    min_samples_leaf=15, l2_regularization=0.1,
                    early_stopping=True, validation_fraction=0.1,
                    n_iter_no_change=30, random_state=42)
    return HistGradientBoostingRegressor(**{**defaults, **kw})


## 2. Load Data

In [ ]:
df = pd.read_parquet('data/features/feature_matrix.parquet')
df = df.sort_values('date').reset_index(drop=True)

X = df[FEATURE_COLS].fillna(0).values.astype(np.float32)

rolling_vol = (
    df[TARGET_COL].rolling(VOL_WINDOW, min_periods=4).std()
    .clip(lower=0.5).bfill().values.astype(np.float32)
)
raw_returns = df[TARGET_COL].values.astype(np.float32)
y           = np.clip(raw_returns / rolling_vol, -10, 10)
dates       = df['date'].values
vols        = rolling_vol

print(f'Feature matrix: {df.shape[0]} rows × {len(FEATURE_COLS)} features')
print(f'Date range: {df["date"].min().date()} → {df["date"].max().date()}')
n_folds = (len(df) - INITIAL_TRAIN_WEEKS) // STEP_WEEKS
print(f'Expected folds: ~{n_folds}  (first OOS: {pd.Timestamp(dates[INITIAL_TRAIN_WEEKS]).date()})')


## 3. Regime Diagnostics
Fit the regime detector once on the pre-2023 training history to inspect
regime characteristics before the full walk-forward evaluation.


In [ ]:
diag_train_end = 260  # first ~5 years for diagnostics
det_diag = OilRegimeDetector(n_states=4)
det_diag.fit(df.iloc[:diag_train_end])

labels_diag = det_diag.predict(df.iloc[:diag_train_end])
print('Transition matrix (probability of staying / switching regime):')
print(det_diag.transition_matrix())
print(f'\nRegime counts: {dict(zip(*np.unique(labels_diag, return_counts=True)))}')
print('\nMean driver values per regime:')
mean_summary = det_diag.regime_summary(df.iloc[:diag_train_end])
print(mean_summary.T.to_string())


In [ ]:
all_labels = det_diag.predict(df.iloc[:diag_train_end])
proba_full = det_diag.predict_proba(df.iloc[:diag_train_end])

REGIME_COLORS = {'supply': '#d95f02', 'demand': '#1b9e77',
                 'macro': '#7570b3', 'risk_off': '#e7298a'}

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Regime probability stacked area
proba_plot = pd.DataFrame(proba_full.values,
    columns=[c.replace('regime_prob_','') for c in proba_full.columns],
    index=df['date'].iloc[:diag_train_end])
axes[0].stackplot(proba_plot.index,
    [proba_plot[r].values for r in REGIME_NAMES],
    labels=REGIME_NAMES, colors=[REGIME_COLORS[r] for r in REGIME_NAMES], alpha=0.8)
axes[0].set_ylabel('Regime probability'); axes[0].set_title('Detected Regime Probabilities')
axes[0].legend(loc='upper left', ncol=4, fontsize=8)

# Dominant regime as coloured bands on price
close = df['close'].iloc[:diag_train_end].values
axes[1].plot(df['date'].iloc[:diag_train_end], close, color='black', linewidth=0.8)
for i in range(len(all_labels) - 1):
    axes[1].axvspan(df['date'].iloc[i], df['date'].iloc[i + 1],
                    alpha=0.25, color=REGIME_COLORS[all_labels[i]])
axes[1].set_ylabel('WTI Close ($/bbl)'); axes[1].set_title('WTI Price with Regime Overlay')

# Realized vol by regime
rv = df['realized_vol_lag1'].iloc[:diag_train_end]
axes[2].plot(df['date'].iloc[:diag_train_end], rv, color='steelblue', linewidth=0.8, label='Realized vol')
for r, c in REGIME_COLORS.items():
    mask = pd.Series(all_labels) == r
    axes[2].scatter(df['date'].iloc[:diag_train_end][mask.values],
                    rv[mask.values], color=c, s=12, alpha=0.7, zorder=3)
axes[2].set_ylabel('Realized vol (%)'); axes[2].set_title('Realized Volatility by Regime')

plt.tight_layout(); plt.show()


## 3. Specialist Mixture-of-Experts Walk-Forward

For each fold we train:
- **1 global model** on all training weeks (expanding window, decay-weighted)
- **4 regime-specific specialists** — one per regime, trained only on weeks where
  that regime dominated (`>= MIN_REGIME_SAMPLES=25`; falls back to global otherwise)

Prediction: `ŷ = GLOBAL_WEIGHT * ŷ_global + (1-GLOBAL_WEIGHT) * Σ_r p(r|x) * ŷ_r`

This lets each specialist learn the dominant driver set for its regime without being
confused by the other regimes' signal patterns.


In [ ]:
fold_records = []
all_preds    = []
last_specialists = {}
last_global = None

for fold, test_start in enumerate(range(INITIAL_TRAIN_WEEKS, len(df), STEP_WEEKS)):
    test_end = min(test_start + STEP_WEEKS, len(df))
    if test_end - test_start < 5:
        break

    df_tr   = df.iloc[:test_start]
    df_te   = df.iloc[test_start:test_end]
    X_tr    = X[:test_start]
    X_te    = X[test_start:test_end]
    y_tr    = y[:test_start]
    act_raw = raw_returns[test_start:test_end]
    vols_te = vols[test_start:test_end]
    dates_te = dates[test_start:test_end]

    # ── Exponential sample weights (global) ──────────────────────────────────
    n     = len(X_tr)
    raw_w = np.array([DECAY_RATE ** (n - 1 - i) for i in range(n)], dtype=np.float32)
    sw    = raw_w / raw_w.mean()

    # ── Regime detection (train only) ─────────────────────────────────────────
    detector = OilRegimeDetector(n_states=4)
    detector.fit(df_tr)
    train_regimes = detector.predict(df_tr)
    test_proba    = detector.predict_proba(df_te)   # (n_test, 4)
    test_regimes  = detector.predict(df_te)

    # ── Global model ──────────────────────────────────────────────────────────
    global_model = _make_model(max_iter=300)
    global_model.fit(X_tr, y_tr, sample_weight=sw)
    last_global = global_model

    # ── Specialist models per regime ──────────────────────────────────────────
    specialist_models = {}
    regime_counts = {}
    for r in REGIME_NAMES:
        mask  = train_regimes == r
        n_r   = int(mask.sum())
        regime_counts[r] = n_r
        if n_r >= MIN_REGIME_SAMPLES:
            # Re-index decay weights within this regime's own timeline
            r_w = np.array([DECAY_RATE ** (n_r - 1 - i) for i in range(n_r)], dtype=np.float32)
            r_w /= r_w.mean()
            m = _make_model(max_iter=300, max_depth=3, min_samples_leaf=10,
                            l2_regularization=0.15, validation_fraction=0.15,
                            n_iter_no_change=25)
            m.fit(X_tr[mask], y_tr[mask], sample_weight=r_w)
            specialist_models[r] = m
        else:
            specialist_models[r] = None  # fallback to global
    last_specialists = specialist_models

    # ── Mixture-of-experts prediction ─────────────────────────────────────────
    pred_global = global_model.predict(X_te)
    pred_expert = np.zeros(test_end - test_start)
    for r in REGIME_NAMES:
        prob  = test_proba[f'regime_prob_{r}'].values
        model = specialist_models[r] if specialist_models[r] is not None else global_model
        pred_expert += prob * model.predict(X_te)

    pred_norm = GLOBAL_WEIGHT * pred_global + (1.0 - GLOBAL_WEIGHT) * pred_expert
    preds_pct = pred_norm * vols_te

    dir_acc_f = float(np.mean(np.sign(preds_pct) == np.sign(act_raw)))
    mae_f     = float(mean_absolute_error(act_raw, preds_pct))
    n_specialists = sum(1 for m in specialist_models.values() if m is not None)

    fold_records.append({'fold': fold, 'train_weeks': n,
        'test_start': pd.Timestamp(dates_te[0]).date(),
        'test_end':   pd.Timestamp(dates_te[-1]).date(),
        'n_test': test_end - test_start, 'mae': mae_f, 'dir_acc': dir_acc_f,
        'n_specialists': n_specialists,
        **{f'n_{r}': regime_counts[r] for r in REGIME_NAMES}})

    for i in range(test_end - test_start):
        all_preds.append({'fold': fold, 'date': pd.Timestamp(dates_te[i]),
                          'actual': float(act_raw[i]), 'pred': float(preds_pct[i]),
                          'regime': test_regimes[i]})

    print(f'Fold {fold:2d}  train={n:3d}w  ',
          f'test={pd.Timestamp(dates_te[0]).date()} → {pd.Timestamp(dates_te[-1]).date()}  ',
          f'dir_acc={dir_acc_f:.1%}  specialists={n_specialists}/4  ',
          f'counts={regime_counts}')

folds_df = pd.DataFrame(fold_records)
results  = pd.DataFrame(all_preds).sort_values('date').reset_index(drop=True)
print(f'\nTotal OOS weeks: {len(results)}  across {len(folds_df)} folds')


## 5. Evaluation

In [ ]:
actual_all = results['actual'].values
pred_all   = results['pred'].values

mae_all  = mean_absolute_error(actual_all, pred_all)
rmse_all = np.sqrt(mean_squared_error(actual_all, pred_all))
dir_all  = np.mean(np.sign(pred_all) == np.sign(actual_all))

print('Specialist MoE Walk-Forward OOS Summary')
print('=' * 56)
print(f'Total OOS weeks      : {len(results)}  across {len(folds_df)} folds')
print(f'Date range           : {results["date"].min().date()} → {results["date"].max().date()}')
print()
print(f'MAE  (all folds)     : {mae_all:.4f}%')
print(f'RMSE (all folds)     : {rmse_all:.4f}%')
print(f'Dir Acc (all folds)  : {dir_all:.1%}')
print(f'Variance ratio       : {pred_all.std() / actual_all.std():.3f}')
print('=' * 56)

print(f'\n{"Threshold":>12}  {"N kept":>6}  {"Dir Acc":>8}  {"Long":>6}  {"Short":>6}')
print('-' * 56)
for thresh in [0.0, 0.5, 1.0, 1.5, 2.0]:
    mask = np.abs(pred_all) >= thresh
    if mask.sum() < 5: break
    da = np.mean(np.sign(pred_all[mask]) == np.sign(actual_all[mask]))
    print(f'{thresh:>11.1f}%  {mask.sum():>6}  {da:>8.1%}  {(pred_all[mask]>0).sum():>6}  {(pred_all[mask]<0).sum():>6}')

print('\nPer-regime directional accuracy (all OOS weeks):')
print(f'{"Regime":>12}  {"N weeks":>8}  {"Dir Acc":>8}  {"MAE":>7}  {"Avg |pred|":>10}')
print('-' * 56)
for r in REGIME_NAMES:
    sub = results[results['regime'] == r]
    if len(sub) == 0: continue
    da_r = np.mean(np.sign(sub['pred'].values) == np.sign(sub['actual'].values))
    mae_r = np.mean(np.abs(sub['pred'].values - sub['actual'].values))
    print(f'{r:>12}  {len(sub):>8}  {da_r:>8.1%}  {mae_r:>7.3f}  {np.abs(sub["pred"]).mean():>10.3f}%')

print('\nPer-fold metrics:')
print(folds_df[['fold','test_start','test_end','n_test','mae','dir_acc','n_specialists']]
      .to_string(index=False, float_format=lambda x: f'{x:.3f}'))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
colors = ['darkorange' if d >= 0.55 else 'steelblue' if d >= 0.50 else 'salmon'
          for d in folds_df['dir_acc']]
axes[0].bar(folds_df['fold'], folds_df['dir_acc'], color=colors, edgecolor='white')
axes[0].axhline(0.5, color='black', lw=1, ls='--')
axes[0].axhline(dir_all, color='red', lw=1.5, label=f'Overall {dir_all:.1%}')
axes[0].set_title('Dir Acc per Fold'); axes[0].set_ylim(0, 1); axes[0].legend()

regime_da = {}
for r in REGIME_NAMES:
    sub = results[results['regime'] == r]
    if len(sub) > 0:
        regime_da[r] = np.mean(np.sign(sub['pred'].values) == np.sign(sub['actual'].values))
axes[1].bar(range(len(regime_da)), regime_da.values(),
            color=[REGIME_COLORS.get(r,'grey') for r in regime_da.keys()], edgecolor='white')
axes[1].axhline(0.5, color='black', lw=1, ls='--')
axes[1].axhline(dir_all, color='red', lw=1.5, ls='--', label=f'Overall {dir_all:.1%}')
axes[1].set_xticks(range(len(regime_da))); axes[1].set_xticklabels(regime_da.keys())
axes[1].set_title('Dir Acc by Regime (OOS)'); axes[1].set_ylim(0, 1); axes[1].legend()
plt.tight_layout(); plt.show()


In [ ]:
# ── Per-fold regime composition ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
bottom = np.zeros(len(folds_df))
for r in REGIME_NAMES:
    counts = folds_df[f'n_{r}'].values
    ax.bar(folds_df['fold'], counts / folds_df['n_test'].values,
           bottom=bottom / folds_df['n_test'].values,
           label=r, color=REGIME_COLORS.get(r, 'grey'), alpha=0.85)
    bottom += counts
ax.axhline(0.5, color='white', lw=0.8, ls=':')
ax.set_xlabel('Fold'); ax.set_ylabel('Fraction of test weeks')
ax.set_title('Regime Composition per Fold')
ax.legend(loc='upper left', ncol=4); plt.tight_layout(); plt.show()

# ── Permutation importance on last global model ───────────────────────────────
if last_global is not None:
    last_fold_start = folds_df['test_start'].iloc[-1]
    oos_mask = results['date'] >= pd.Timestamp(last_fold_start)
    oos_sub  = results[oos_mask]
    X_oos = df[df['date'] >= pd.Timestamp(last_fold_start)][FEATURE_COLS].fillna(0).values.astype(np.float32)
    y_oos = raw_returns[df[df['date'] >= pd.Timestamp(last_fold_start)].index]
    if len(X_oos) >= 10:
        pi = permutation_importance(last_global, X_oos, y_oos, n_repeats=20,
                                     scoring='neg_mean_absolute_error', random_state=42)
        top_idx = np.argsort(pi.importances_mean)[::-1][:15]
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.barh([FEATURE_COLS[i] for i in top_idx[::-1]],
                pi.importances_mean[top_idx[::-1]], xerr=pi.importances_std[top_idx[::-1]],
                color='steelblue', capsize=3)
        ax.set_title('Permutation Importance — Global Model (last fold)')
        ax.axvline(0, color='black', lw=0.8)
        plt.tight_layout(); plt.show()


## 6. Strategy Comparison

In [ ]:
test_df = results.copy().sort_values('date').reset_index(drop=True)

test_df['signal_naive']    = np.where(test_df['pred'] > 0, 1, 0)
test_df['signal_filtered'] = np.where(
    test_df['pred'] >=  CONFIDENCE_THRESHOLD,  1,
    np.where(test_df['pred'] <= -CONFIDENCE_THRESHOLD, -1, 0))

# Regime-filtered: avoid trading in macro regime (lowest dir acc in OOS)
avoid_regime = 'macro'
test_df['signal_regime_filtered'] = test_df.apply(
    lambda r: 0 if r['regime'] == avoid_regime else r['signal_filtered'], axis=1)

test_df['ret_naive']    = test_df['signal_naive']           * test_df['actual']
test_df['ret_filtered'] = test_df['signal_filtered']        * test_df['actual']
test_df['ret_rf']       = test_df['signal_regime_filtered'] * test_df['actual']

test_df['cum_bh']       = (1 + test_df['actual']        / 100).cumprod()
test_df['cum_naive']    = (1 + test_df['ret_naive']     / 100).cumprod()
test_df['cum_filtered'] = (1 + test_df['ret_filtered']  / 100).cumprod()
test_df['cum_rf']       = (1 + test_df['ret_rf']        / 100).cumprod()

print(f'Walk-Forward Specialist MoE Strategy',
      f'({test_df["date"].min().date()} → {test_df["date"].max().date()})')
print(f'{"Strategy":<38}  {"Trades":>7}  {"Long":>6}  {"Short":>6}  {"Dir Acc":>8}  {"Final":>7}')
print('-' * 82)

for sig_col, ret_col, label in [
    ('signal_naive',           'cum_naive',    'Naive'),
    ('signal_filtered',        'cum_filtered', f'Filtered (|pred|≥{CONFIDENCE_THRESHOLD}%)'),
    ('signal_regime_filtered', 'cum_rf',       f'Filtered + skip {avoid_regime}'),
]:
    mask = test_df[sig_col] != 0
    if mask.sum() == 0:
        print(f'{label:<38}  {"—":>7}'); continue
    da = np.mean(np.sign(test_df.loc[mask, 'pred']) == np.sign(test_df.loc[mask, 'actual']))
    nl = (test_df[sig_col] ==  1).sum()
    ns = (test_df[sig_col] == -1).sum()
    fin = test_df[ret_col].iloc[-1]
    print(f'{label:<38}  {mask.sum():>7}  {nl:>6}  {ns:>6}  {da:>8.1%}  {fin:>7.3f}x')
print(f'{"Buy & Hold":<38}  {"—":>7}  {"—":>6}  {"—":>6}  {"—":>8}  {test_df["cum_bh"].iloc[-1]:>7.3f}x')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test_df['date'], test_df['cum_bh'],       label='Buy & Hold',               color='steelblue', ls='--')
ax.plot(test_df['date'], test_df['cum_naive'],    label='Naive',                    color='grey',      ls=':')
ax.plot(test_df['date'], test_df['cum_filtered'], label=f'Filtered (≥{CONFIDENCE_THRESHOLD}%)', color='darkorange', lw=1.5)
ax.plot(test_df['date'], test_df['cum_rf'],       label=f'Filtered + skip {avoid_regime}', color='darkgreen', lw=2)
for _, row in folds_df.iterrows():
    ax.axvspan(pd.Timestamp(row['test_start']), pd.Timestamp(row['test_end']),
               alpha=0.03, color='grey')
ax.axhline(1, color='black', lw=0.5, ls=':')
ax.set_title('Specialist MoE — Cumulative Strategy Return')
ax.set_ylabel('Growth of $1'); ax.legend(); plt.tight_layout(); plt.show()
